# Chapter 7: Inference, Evaluation & Reasoning in Vision-Language Models

**What you'll build:** The full inference pipeline — from raw pixels to generated text — along with reasoning modes, structured outputs, evaluation harnesses, and hallucination analysis.

| Section | Topic |
|---------|-------|
| 7.1 | Inference pipeline: image → encode → project → concat → generate |
| 7.2 | Decoding strategies: greedy, beam search, nucleus sampling |
| 7.3 | Streaming generation (token-by-token output) |
| 7.4 | Thinking mode: chain-of-thought reasoning over images |
| 7.5 | Visual grounding: bounding boxes as structured output |
| 7.6 | Evaluation benchmarks |
| 7.7 | Hallucination analysis and mitigation |
| 7.8 | Build a Gradio UI for interactive multimodal chat |

**Prerequisites:** Familiarity with transformers, attention, backpropagation, and basic RL.
**Framework:** PyTorch only. We implement core inference logic from scratch; peripheral utilities come from HuggingFace/standard libraries.

---
## Setup & Dependencies

In [ ]:
!pip install -q torch torchvision transformers pillow matplotlib gradio rouge-score nltk

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict
import time
import json
import math
import warnings

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

---
# 1) Inference Pipeline: Image → Encode → Project → Concat → Generate

## 1.1 Motivation & Architecture Overview

A Vision-Language Model (VLM) inference pipeline has **four distinct stages**:

1. **Visual Encoding** — A frozen or fine-tuned vision encoder (e.g., ViT) maps a raw image into a sequence of patch embeddings.
2. **Modality Projection** — A learned MLP bridges the vision feature space into the language model's embedding space.
3. **Sequence Construction** — Visual tokens are concatenated with text token embeddings, forming a unified multimodal sequence.
4. **Autoregressive Generation** — A causal language model generates output tokens conditioned on the fused sequence.

```
Sample Input:
  image: [3, 224, 224] RGB tensor (a photo of a dog on a beach)
  prompt: "Describe this image in detail."

Sample Output:
  "A golden retriever is running along a sandy beach with waves in the background."
```

**Why this design?** Separate encoding allows reuse of powerful pretrained vision backbones (CLIP-ViT, SigLIP) while the projection layer handles the distributional gap between vision and language feature spaces. The concat-then-attend pattern lets the LM's self-attention naturally fuse cross-modal information without architectural changes.

## 1.2 Vision Encoder (Patch Embedding + Transformer)

We implement a minimal ViT encoder from scratch. The image is split into non-overlapping patches, linearly projected, and processed by transformer blocks.

```
Input:  (batch_num, 3, 224, 224)  — RGB image
Output: (batch_num, num_patches, embed_dim)  — patch embeddings

With patch_size=16: num_patches = (224/16)^2 = 196
```

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image into a sequence of flattened patch embeddings.

    Uses a single conv2d with kernel_size=stride=patch_size, which is
    mathematically equivalent to extracting non-overlapping patches and
    linearly projecting each one, but far more efficient.
    """

    def __init__(self, img_size: int = 224, patch_size: int = 16, in_channels: int = 3, embed_dim: int = 256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size

        # (batch_num, 3, 224, 224) → (batch_num, embed_dim, 14, 14)
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # Learnable positional embeddings for each patch location
        # (1, num_patches, embed_dim)
        self.position_embeddings = nn.Parameter(torch.randn(1, self.num_patches, embed_dim) * 0.02)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # (batch_num, 3, img_size, img_size) → (batch_num, embed_dim, grid, grid)
        x = self.projection(pixel_values)

        # (batch_num, embed_dim, grid, grid) → (batch_num, num_patches, embed_dim)
        x = x.flatten(2).transpose(1, 2)

        # Add positional information so the model knows spatial layout
        x = x + self.position_embeddings
        return x


class VisionTransformerBlock(nn.Module):
    """Standard pre-norm transformer block for the vision encoder."""

    def __init__(self, embed_dim: int = 256, num_heads: int = 8, mlp_ratio: float = 4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)

        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Self-attention with residual
        # (batch_num, num_patches, embed_dim) → (batch_num, num_patches, embed_dim)
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out

        # Feed-forward with residual
        # (batch_num, num_patches, embed_dim) → (batch_num, num_patches, embed_dim)
        x = x + self.mlp(self.norm2(x))
        return x


class VisionEncoder(nn.Module):
    """Minimal ViT: patch embed → N transformer blocks → layer norm."""

    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        embed_dim: int = 256,
        num_layers: int = 4,
        num_heads: int = 8,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.blocks = nn.ModuleList(
            [VisionTransformerBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # (batch_num, 3, img_size, img_size) → (batch_num, num_patches, embed_dim)
        x = self.patch_embed(pixel_values)
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


# Verify shapes
encoder = VisionEncoder().to(DEVICE)
dummy_img = torch.randn(2, 3, 224, 224, device=DEVICE)
vis_features = encoder(dummy_img)
print(f"Vision encoder: {dummy_img.shape} → {vis_features.shape}")
print(f"  num_patches = {vis_features.shape[1]}, embed_dim = {vis_features.shape[2]}")

## 1.3 Modality Projection Layer

The vision encoder outputs features in its own representation space. The language model expects inputs in *its* embedding space. The projection layer is a small MLP that bridges this gap.

```
Input:  (batch_num, num_patches, vision_dim)   — from ViT
Output: (batch_num, num_patches, model_dim)     — compatible with LM embeddings
```

**Design choice:** We use a two-layer MLP with GELU, following LLaVA-1.5. A single linear layer works but the nonlinearity helps the model learn a richer cross-modal mapping. Some architectures (Qwen2-VL) additionally downsample the visual token count here to reduce KV-cache pressure.

In [ ]:
class ModalityProjection(nn.Module):
    """Two-layer MLP that maps vision features into the LM embedding space.

    Optionally applies spatial pooling to reduce the number of visual tokens,
    trading spatial resolution for inference speed.
    """

    def __init__(self, vision_dim: int = 256, model_dim: int = 512, pool_factor: int = 1):
        super().__init__()
        self.pool_factor = pool_factor

        # If pooling, the input dimension grows by pool_factor^2 due to reshaping
        input_dim = vision_dim * (pool_factor ** 2) if pool_factor > 1 else vision_dim

        # (batch_num, num_vis_tokens, vision_dim) → (batch_num, num_vis_tokens, model_dim)
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, model_dim),
            nn.GELU(),
            nn.Linear(model_dim, model_dim),
        )

    def _spatial_pool(self, x: torch.Tensor) -> torch.Tensor:
        """Merge adjacent patches to reduce token count.

        Reshapes the grid so that pool_factor×pool_factor patches are
        concatenated into a single token. Reduces num_patches by pool_factor^2.
        """
        batch_num, num_patches, dim = x.shape
        grid_size = int(math.sqrt(num_patches))
        new_grid = grid_size // self.pool_factor

        # (batch_num, grid, grid, dim) → reshape into pooled grid
        x = x.view(batch_num, grid_size, grid_size, dim)
        x = x.view(batch_num, new_grid, self.pool_factor, new_grid, self.pool_factor, dim)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
        x = x.view(batch_num, new_grid * new_grid, dim * self.pool_factor ** 2)
        return x

    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        if self.pool_factor > 1:
            vision_features = self._spatial_pool(vision_features)
        return self.mlp(vision_features)


# Without pooling: 196 tokens preserved
proj_no_pool = ModalityProjection(vision_dim=256, model_dim=512, pool_factor=1).to(DEVICE)
projected_no_pool = proj_no_pool(vis_features)
print(f"No pooling:  {vis_features.shape} → {projected_no_pool.shape}")

# With 2x pooling: 196 → 49 tokens (4x reduction)
proj_pool2 = ModalityProjection(vision_dim=256, model_dim=512, pool_factor=2).to(DEVICE)
projected_pool2 = proj_pool2(vis_features)
print(f"Pool 2x:     {vis_features.shape} → {projected_pool2.shape}")

## 1.4 Causal Language Model (Decoder)

We build a minimal GPT-style decoder that accepts the concatenated visual + text tokens and generates output autoregressively. The key constraint: **causal masking** ensures each position only attends to earlier positions.

```
Input:  (batch_num, seq_len, model_dim)  — concatenated [visual_tokens; text_tokens]
Output: (batch_num, seq_len, vocab_size) — next-token logits
```

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with causal (autoregressive) masking."""

    def __init__(self, model_dim: int = 512, num_heads: int = 8, max_len: int = 1024):
        super().__init__()
        assert model_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads

        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out_proj = nn.Linear(model_dim, model_dim)

        # Pre-compute causal mask: upper-triangular = -inf
        # (max_len, max_len)
        causal_mask = torch.triu(torch.ones(max_len, max_len), diagonal=1).bool()
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, seq_len, model_dim = x.shape

        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, 3 * model_dim)
        qkv = self.qkv(x)

        # Split into Q, K, V each of shape (batch_num, num_heads, seq_len, head_dim)
        qkv = qkv.view(batch_num, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Scaled dot-product attention
        # (batch_num, num_heads, seq_len, head_dim) × (batch_num, num_heads, head_dim, seq_len)
        # → (batch_num, num_heads, seq_len, seq_len)
        scale = math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / scale

        # Apply causal mask: prevent attending to future tokens
        mask = self.causal_mask[:seq_len, :seq_len]
        attn_weights = attn_weights.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        attn_weights = F.softmax(attn_weights, dim=-1)

        # (batch_num, num_heads, seq_len, seq_len) × (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, seq_len, head_dim)
        attn_output = torch.matmul(attn_weights, v)

        # Merge heads: (batch_num, seq_len, model_dim)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_num, seq_len, model_dim)
        return self.out_proj(attn_output)


class DecoderBlock(nn.Module):
    """Pre-norm transformer decoder block."""

    def __init__(self, model_dim: int = 512, num_heads: int = 8, max_len: int = 1024):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.attn = CausalSelfAttention(model_dim, num_heads, max_len)
        self.norm2 = nn.LayerNorm(model_dim)
        self.mlp = nn.Sequential(
            nn.Linear(model_dim, 4 * model_dim),
            nn.GELU(),
            nn.Linear(4 * model_dim, model_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class CausalLM(nn.Module):
    """Minimal GPT-style causal language model.

    Accepts pre-embedded inputs (not raw token ids) so that the
    VLM pipeline can feed concatenated vision+text embeddings directly.
    """

    def __init__(
        self,
        vocab_size: int = 10000,
        model_dim: int = 512,
        num_layers: int = 4,
        num_heads: int = 8,
        max_len: int = 1024,
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, model_dim)
        self.position_embedding = nn.Embedding(max_len, model_dim)
        self.blocks = nn.ModuleList(
            [DecoderBlock(model_dim, num_heads, max_len) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(model_dim)

        # Output head: project back to vocabulary
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, vocab_size)
        self.lm_head = nn.Linear(model_dim, vocab_size, bias=False)

    def embed_tokens(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Convert token ids to embeddings with positional encoding."""
        seq_len = token_ids.shape[1]
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        return self.token_embedding(token_ids) + self.position_embedding(positions)

    def forward_embeddings(self, embeddings: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        """Run decoder on pre-constructed embeddings.

        This is the entry point for the VLM pipeline, which feeds
        concatenated [visual_emb | text_emb] directly.
        """
        seq_len = embeddings.shape[1]
        positions = torch.arange(start_pos, start_pos + seq_len, device=embeddings.device).unsqueeze(0)
        x = embeddings + self.position_embedding(positions)

        for block in self.blocks:
            x = block(x)

        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, vocab_size)
        return self.lm_head(self.norm(x))


lm = CausalLM(vocab_size=10000, model_dim=512).to(DEVICE)
dummy_emb = torch.randn(2, 50, 512, device=DEVICE)
logits = lm.forward_embeddings(dummy_emb)
print(f"LM decoder: {dummy_emb.shape} → {logits.shape}")

## 1.5 The Complete VLM Inference Pipeline

We now wire all components together into a single `VLMPipeline` class that takes a raw image and text prompt and produces generated text.

```
                    ┌──────────────┐
    image ────────► │ VisionEncoder │──► visual_tokens
                    └──────────────┘        │
                                            ▼
                    ┌───────────────────┐   │
                    │ ModalityProjection │◄──┘
                    └───────────────────┘
                            │
               projected_visual_tokens
                            │
                            ▼
              ┌─────────────────────────┐
    prompt ──►│ Concat [vis | text_emb] │
              └─────────────────────────┘
                            │
                            ▼
                    ┌───────────┐
                    │ CausalLM  │──► output tokens
                    └───────────┘
```

In [ ]:
class VLMPipeline(nn.Module):
    """End-to-end vision-language model for inference.

    Connects vision encoder → projection → causal LM and provides
    a unified generate() interface.
    """

    def __init__(
        self,
        vision_encoder: VisionEncoder,
        projection: ModalityProjection,
        language_model: CausalLM,
        vocab_size: int = 10000,
        eos_token_id: int = 2,
    ):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.projection = projection
        self.language_model = language_model
        self.vocab_size = vocab_size
        self.eos_token_id = eos_token_id

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Image → projected visual tokens ready for the LM."""
        # (batch_num, 3, img_size, img_size) → (batch_num, num_patches, vision_dim)
        vis_features = self.vision_encoder(pixel_values)
        # (batch_num, num_patches, vision_dim) → (batch_num, num_vis_tokens, model_dim)
        return self.projection(vis_features)

    def build_multimodal_input(
        self, visual_tokens: torch.Tensor, prompt_ids: torch.Tensor
    ) -> torch.Tensor:
        """Concatenate visual tokens with text embeddings."""
        # (batch_num, prompt_len, model_dim)
        text_embeddings = self.language_model.token_embedding(prompt_ids)

        # (batch_num, num_vis_tokens + prompt_len, model_dim)
        return torch.cat([visual_tokens, text_embeddings], dim=1)

    @torch.no_grad()
    def generate(
        self,
        pixel_values: torch.Tensor,
        prompt_ids: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 1.0,
        strategy: str = "greedy",
        top_p: float = 0.9,
        top_k: int = 50,
    ) -> torch.Tensor:
        """Full generation loop. Strategy options: greedy, sampling, nucleus."""
        visual_tokens = self.encode_image(pixel_values)
        input_embeds = self.build_multimodal_input(visual_tokens, prompt_ids)

        generated_ids = []
        current_embeds = input_embeds

        for _ in range(max_new_tokens):
            # (batch_num, current_len, model_dim) → (batch_num, current_len, vocab_size)
            logits = self.language_model.forward_embeddings(current_embeds)

            # Take logits at the last position
            # (batch_num, vocab_size)
            next_logits = logits[:, -1, :] / temperature

            # Apply decoding strategy
            next_token = self._decode(next_logits, strategy, top_p, top_k)
            generated_ids.append(next_token)

            # Stop if EOS generated (for batch_size=1)
            if next_token.item() == self.eos_token_id:
                break

            # Embed new token and append to sequence
            # (batch_num, 1, model_dim)
            next_embed = self.language_model.token_embedding(next_token.unsqueeze(1))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)

        return torch.cat(generated_ids, dim=-1) if generated_ids else torch.tensor([], device=pixel_values.device)

    def _decode(
        self, logits: torch.Tensor, strategy: str, top_p: float, top_k: int
    ) -> torch.Tensor:
        """Select next token based on decoding strategy."""
        if strategy == "greedy":
            return logits.argmax(dim=-1)
        elif strategy == "sampling":
            probs = F.softmax(logits, dim=-1)
            return torch.multinomial(probs, 1).squeeze(-1)
        elif strategy == "nucleus":
            return self._nucleus_sample(logits, top_p)
        elif strategy == "top_k":
            return self._top_k_sample(logits, top_k)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

    def _nucleus_sample(self, logits: torch.Tensor, top_p: float) -> torch.Tensor:
        """Nucleus (top-p) sampling: sample from the smallest set whose cumulative probability >= top_p."""
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Zero out tokens beyond the nucleus
        sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
        sorted_logits[sorted_mask] = float("-inf")

        probs = F.softmax(sorted_logits, dim=-1)
        sampled_index = torch.multinomial(probs, 1)
        return sorted_indices.gather(-1, sampled_index).squeeze(-1)

    def _top_k_sample(self, logits: torch.Tensor, k: int) -> torch.Tensor:
        """Top-K sampling: restrict to the K highest-probability tokens."""
        top_k_logits, top_k_indices = torch.topk(logits, k, dim=-1)
        probs = F.softmax(top_k_logits, dim=-1)
        sampled_index = torch.multinomial(probs, 1)
        return top_k_indices.gather(-1, sampled_index).squeeze(-1)


# Instantiate the full pipeline
vision_enc = VisionEncoder(img_size=224, patch_size=16, embed_dim=256, num_layers=4).to(DEVICE)
proj = ModalityProjection(vision_dim=256, model_dim=512).to(DEVICE)
lm_decoder = CausalLM(vocab_size=10000, model_dim=512, num_layers=4).to(DEVICE)

vlm = VLMPipeline(vision_enc, proj, lm_decoder).to(DEVICE)

# Test end-to-end generation
test_image = torch.randn(1, 3, 224, 224, device=DEVICE)
test_prompt = torch.randint(0, 10000, (1, 10), device=DEVICE)

output_ids = vlm.generate(test_image, test_prompt, max_new_tokens=20, strategy="greedy")
print(f"Input image: {test_image.shape}")
print(f"Input prompt: {test_prompt.shape} (10 tokens)")
print(f"Generated: {output_ids.shape} token ids")
print(f"Token ids: {output_ids.tolist()}")

---
# 2) Decoding Strategies: Greedy, Beam Search, Nucleus Sampling

## 2.1 Overview

Given a language model that outputs a distribution over the vocabulary at each step, **decoding strategies** determine how we convert those distributions into actual token sequences. Each strategy represents a different tradeoff:

| Strategy | Diversity | Quality | Speed | Use Case |
|----------|-----------|---------|-------|----------|
| Greedy | Low | Decent | Fast | Deterministic factual answers |
| Beam Search | Low-Med | High | Slow | Translation, captioning |
| Top-K | Medium | Good | Fast | Creative text |
| Nucleus (Top-p) | High | Good | Fast | Open-ended generation |
| Temperature | Tunable | Tunable | Fast | Control randomness |

```
Sample: Given logits = [2.0, 1.0, 0.5, 0.1, -1.0] over vocab [A, B, C, D, E]

Greedy:     → always picks A (highest logit)
Beam (k=2): → explores {A, B} in parallel, picks best complete sequence
Nucleus:    → computes cumulative prob, samples from {A, B, C} if top_p=0.9
```

## 2.2 Beam Search — From Scratch

Beam search maintains `beam_width` candidate sequences at each step, expanding each by the top candidates and keeping only the overall best. This explores more of the search space than greedy while remaining tractable.

**Key insight:** Beam search with beam_width=1 reduces to greedy decoding. Larger beams improve output quality but increase computation linearly.

In [ ]:
@dataclass
class BeamHypothesis:
    """A single hypothesis (partial sequence) in beam search."""
    token_ids: List[int]
    score: float  # log-probability of the sequence so far
    finished: bool = False


class BeamSearchDecoder:
    """Beam search over an autoregressive language model.

    Implements length-normalized scoring to avoid bias toward shorter sequences.
    """

    def __init__(
        self,
        beam_width: int = 4,
        max_new_tokens: int = 50,
        eos_token_id: int = 2,
        length_penalty: float = 0.6,
    ):
        self.beam_width = beam_width
        self.max_new_tokens = max_new_tokens
        self.eos_token_id = eos_token_id
        self.length_penalty = length_penalty

    def _length_normalized_score(self, log_prob: float, length: int) -> float:
        """Normalize score by sequence length to prevent short-sequence bias.

        Wu et al. (2016) penalty: score / ((5 + length)/(5 + 1))^alpha
        """
        penalty = ((5.0 + length) / 6.0) ** self.length_penalty
        return log_prob / penalty

    @torch.no_grad()
    def search(
        self,
        model: CausalLM,
        initial_embeds: torch.Tensor,
    ) -> List[BeamHypothesis]:
        """Run beam search given initial embeddings (visual + prompt).

        Returns all beams sorted by length-normalized score.
        """
        beams = [BeamHypothesis(token_ids=[], score=0.0)]
        finished_beams = []

        for step in range(self.max_new_tokens):
            all_candidates = []

            for beam in beams:
                if beam.finished:
                    finished_beams.append(beam)
                    continue

                # Build current input: initial embeddings + generated tokens so far
                if beam.token_ids:
                    gen_ids = torch.tensor([beam.token_ids], device=initial_embeds.device)
                    gen_embeds = model.token_embedding(gen_ids)
                    current_embeds = torch.cat([initial_embeds, gen_embeds], dim=1)
                else:
                    current_embeds = initial_embeds

                # (1, current_len, model_dim) → (1, current_len, vocab_size)
                logits = model.forward_embeddings(current_embeds)

                # (vocab_size,)
                next_log_probs = F.log_softmax(logits[0, -1, :], dim=-1)

                # Keep top beam_width candidates from this beam
                topk_log_probs, topk_ids = torch.topk(next_log_probs, self.beam_width)

                for i in range(self.beam_width):
                    token_id = topk_ids[i].item()
                    new_score = beam.score + topk_log_probs[i].item()
                    new_ids = beam.token_ids + [token_id]
                    is_finished = token_id == self.eos_token_id

                    all_candidates.append(
                        BeamHypothesis(token_ids=new_ids, score=new_score, finished=is_finished)
                    )

            if not all_candidates:
                break

            # Select top beam_width candidates by length-normalized score
            all_candidates.sort(
                key=lambda h: self._length_normalized_score(h.score, len(h.token_ids)),
                reverse=True,
            )
            beams = all_candidates[: self.beam_width]

            # Stop if all beams are finished
            if all(b.finished for b in beams):
                finished_beams.extend(beams)
                break

        # Add any remaining active beams
        finished_beams.extend([b for b in beams if not b.finished])

        # Sort by normalized score
        finished_beams.sort(
            key=lambda h: self._length_normalized_score(h.score, max(len(h.token_ids), 1)),
            reverse=True,
        )
        return finished_beams


# Demonstrate beam search
beam_decoder = BeamSearchDecoder(beam_width=3, max_new_tokens=15)
initial_emb = torch.randn(1, 200, 512, device=DEVICE)

beams = beam_decoder.search(lm_decoder, initial_emb)
print(f"Beam search returned {len(beams)} hypotheses:")
for i, beam in enumerate(beams[:3]):
    norm_score = beam_decoder._length_normalized_score(beam.score, len(beam.token_ids))
    print(f"  Beam {i}: {len(beam.token_ids)} tokens, raw_score={beam.score:.2f}, norm_score={norm_score:.2f}")
    print(f"           ids={beam.token_ids[:10]}...")

## 2.3 Comparing Decoding Strategies

We now compare all strategies on the same input, visualizing how the probability distributions differ.

In [ ]:
def visualize_decoding_strategies(logits: torch.Tensor, top_k: int = 10):
    """Compare how different decoding strategies select tokens from the same logits."""
    vocab_size = logits.shape[-1]
    probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()

    # Get top-k tokens for visualization
    top_indices = np.argsort(probs)[-top_k:][::-1]
    top_probs = probs[top_indices]

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    labels = [f"tok_{i}" for i in top_indices]

    # Greedy: all mass on argmax
    greedy_probs = np.zeros_like(top_probs)
    greedy_probs[0] = 1.0
    axes[0].bar(labels, greedy_probs, color="steelblue")
    axes[0].set_title("Greedy\n(deterministic)")
    axes[0].set_ylim(0, 1.1)

    # Temperature = 0.5 (sharper)
    temp_probs = F.softmax(logits.squeeze() / 0.5, dim=-1).cpu().numpy()[top_indices]
    axes[1].bar(labels, temp_probs, color="coral")
    axes[1].set_title("Temperature=0.5\n(sharper)")
    axes[1].set_ylim(0, 1.1)

    # Top-K (k=5)
    topk_probs = top_probs.copy()
    topk_probs[5:] = 0
    topk_probs = topk_probs / topk_probs.sum()
    axes[2].bar(labels, topk_probs, color="mediumseagreen")
    axes[2].set_title("Top-K (K=5)\n(truncated)")
    axes[2].set_ylim(0, 1.1)

    # Nucleus (top-p = 0.9)
    sorted_probs = np.sort(probs)[::-1]
    cumsum = np.cumsum(sorted_probs)
    cutoff_idx = np.searchsorted(cumsum, 0.9) + 1
    nucleus_probs = top_probs.copy()
    nucleus_probs[cutoff_idx:] = 0
    if nucleus_probs.sum() > 0:
        nucleus_probs = nucleus_probs / nucleus_probs.sum()
    axes[3].bar(labels, nucleus_probs, color="orchid")
    axes[3].set_title(f"Nucleus (p=0.9)\n(top {cutoff_idx} tokens)")
    axes[3].set_ylim(0, 1.1)

    for ax in axes:
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("Probability")

    plt.suptitle("Decoding Strategy Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Generate sample logits and visualize
sample_logits = torch.randn(1, 10000, device=DEVICE)
sample_logits[0, 42] = 5.0
sample_logits[0, 100] = 4.0
sample_logits[0, 7] = 3.5
visualize_decoding_strategies(sample_logits)

---
# 3) Streaming Generation (Token-by-Token Output)

## 3.1 Motivation

In production, users expect to see output **as it's generated** rather than waiting for the full sequence. Streaming generation yields tokens one at a time, enabling responsive UIs.

```
Non-streaming (user waits 5 seconds):
  → "A golden retriever is running along a sandy beach."

Streaming (tokens appear progressively):
  → "A" ... "golden" ... "retriever" ... "is" ... "running" ...
```

**Implementation:** We use Python generators (`yield`) to produce tokens lazily. In a real deployment this would feed into a WebSocket or Server-Sent Events (SSE) endpoint.

In [ ]:
class StreamingVLM:
    """Wrapper that adds streaming generation to the VLM pipeline.

    Uses Python generators so each token is yielded immediately after
    it's produced, without waiting for the full sequence.
    """

    def __init__(self, vlm_pipeline: VLMPipeline):
        self.vlm = vlm_pipeline

    @torch.no_grad()
    def stream_generate(
        self,
        pixel_values: torch.Tensor,
        prompt_ids: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 0.8,
        top_p: float = 0.9,
    ):
        """Generator that yields (token_id, token_prob, elapsed_ms) tuples."""
        visual_tokens = self.vlm.encode_image(pixel_values)
        current_embeds = self.vlm.build_multimodal_input(visual_tokens, prompt_ids)

        start_time = time.time()

        for step in range(max_new_tokens):
            step_start = time.time()

            # Forward pass
            # (1, current_len, model_dim) → (1, current_len, vocab_size)
            logits = self.vlm.language_model.forward_embeddings(current_embeds)
            next_logits = logits[:, -1, :] / temperature

            # Nucleus sampling for diverse streaming output
            token_id = self.vlm._nucleus_sample(next_logits, top_p)
            token_prob = F.softmax(next_logits, dim=-1)[0, token_id].item()

            elapsed_ms = (time.time() - step_start) * 1000

            # Yield immediately — caller processes this token before we continue
            yield token_id.item(), token_prob, elapsed_ms

            if token_id.item() == self.vlm.eos_token_id:
                break

            # Extend sequence with the new token
            next_embed = self.vlm.language_model.token_embedding(token_id.unsqueeze(0).unsqueeze(0))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)


# Demonstrate streaming
streamer = StreamingVLM(vlm)

print("Streaming generation (token-by-token):")
print("-" * 60)
total_tokens = 0
total_time = 0

for token_id, prob, ms in streamer.stream_generate(
    test_image, test_prompt, max_new_tokens=20, temperature=0.8
):
    total_tokens += 1
    total_time += ms
    print(f"  token={token_id:5d}  prob={prob:.4f}  latency={ms:.1f}ms")

print("-" * 60)
print(f"Generated {total_tokens} tokens in {total_time:.0f}ms")
print(f"Average: {total_time / max(total_tokens, 1):.1f}ms/token")

## 3.2 KV-Cache for Efficient Streaming

The naive implementation above recomputes attention over the **entire** sequence at every step. With KV-caching, we store the key and value projections from all previous positions and only compute the new token's contribution.

```
Without KV-cache: step t costs O(t²) — reprocesses everything
With KV-cache:    step t costs O(t)  — only new token attends to cached KV

For 1000-token generation: ~500x speedup at the final steps.
```

In [ ]:
class KVCache:
    """Stores cached key-value pairs for efficient autoregressive generation.

    Each layer has its own KV cache that grows as tokens are generated.
    """

    def __init__(self, num_layers: int, num_heads: int, head_dim: int, device: torch.device):
        self.num_layers = num_layers
        # Each entry: (batch_num, num_heads, cached_len, head_dim)
        self.key_cache = [None] * num_layers
        self.value_cache = [None] * num_layers
        self.device = device

    def update(self, layer_idx: int, new_key: torch.Tensor, new_value: torch.Tensor):
        """Append new key/value to the cache for a given layer."""
        if self.key_cache[layer_idx] is None:
            self.key_cache[layer_idx] = new_key
            self.value_cache[layer_idx] = new_value
        else:
            # (batch_num, num_heads, old_len, head_dim) + (batch_num, num_heads, 1, head_dim)
            # → (batch_num, num_heads, old_len + 1, head_dim)
            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], new_key], dim=2)
            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], new_value], dim=2)

    def get(self, layer_idx: int) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        return self.key_cache[layer_idx], self.value_cache[layer_idx]

    @property
    def seq_len(self) -> int:
        if self.key_cache[0] is None:
            return 0
        return self.key_cache[0].shape[2]

    def reset(self):
        self.key_cache = [None] * self.num_layers
        self.value_cache = [None] * self.num_layers


class CachedCausalAttention(nn.Module):
    """Causal self-attention that uses KV-cache during generation.

    During prefill, processes all tokens and populates the cache.
    During generation, processes only the new token and attends to cached KV.
    """

    def __init__(self, model_dim: int = 512, num_heads: int = 8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out_proj = nn.Linear(model_dim, model_dim)

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[KVCache] = None,
        layer_idx: int = 0,
    ) -> torch.Tensor:
        batch_num, seq_len, model_dim = x.shape

        # (batch_num, seq_len, 3 * model_dim)
        qkv = self.qkv(x)
        qkv = qkv.view(batch_num, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        if kv_cache is not None:
            # Store new KV in cache
            kv_cache.update(layer_idx, k, v)
            # Use full cached KV for attention
            k, v = kv_cache.get(layer_idx)

        # Scaled dot-product attention
        # q: (batch_num, num_heads, seq_len, head_dim)
        # k: (batch_num, num_heads, full_len, head_dim)
        scale = math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / scale

        # Causal mask only needed during prefill (seq_len > 1)
        if seq_len > 1:
            full_len = k.shape[2]
            mask = torch.triu(
                torch.ones(seq_len, full_len, device=x.device),
                diagonal=full_len - seq_len + 1
            ).bool()
            attn_weights = attn_weights.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, v)

        # (batch_num, seq_len, model_dim)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_num, seq_len, model_dim)
        return self.out_proj(attn_output)


# Demonstrate KV-cache memory savings
cache = KVCache(num_layers=4, num_heads=8, head_dim=64, device=DEVICE)
cached_attn = CachedCausalAttention(model_dim=512, num_heads=8).to(DEVICE)

# Prefill: process 200 tokens at once
prefill_input = torch.randn(1, 200, 512, device=DEVICE)
_ = cached_attn(prefill_input, kv_cache=cache, layer_idx=0)
print(f"After prefill: cache holds {cache.seq_len} positions")

# Generate: process 1 token at a time
for i in range(5):
    new_token = torch.randn(1, 1, 512, device=DEVICE)
    _ = cached_attn(new_token, kv_cache=cache, layer_idx=0)
    print(f"  Step {i+1}: cache holds {cache.seq_len} positions")

---
# 4) Thinking Mode: Chain-of-Thought Reasoning Over Images

## 4.1 Motivation

Standard VLMs generate answers directly, which works for simple queries but fails on multi-step reasoning (math, spatial logic, complex counting). **Thinking mode** wraps the response in explicit chain-of-thought (CoT) reasoning, delimited by `<think>...</think>` tags.

This is the approach used by Qwen3-VL-Thinking (2025) and GLM-4.1V-Thinking.

```
Input:  Image of a geometry diagram + "What is the area of the shaded region?"

Non-thinking output:
  "The area is approximately 12 square units."

Thinking output:
  <think>
  The diagram shows a circle inscribed in a square.
  The square has side length 4, so area = 16.
  The circle has radius 2, so area = π × 4 ≈ 12.57.
  The shaded region is the square minus the circle.
  Area = 16 - 12.57 = 3.43 square units.
  </think>
  The shaded region has an area of approximately 3.43 square units.
```

**Key design:** The `<think>` tokens are part of the vocabulary and are trained with supervised fine-tuning on reasoning traces. At inference, we can either force thinking mode (by seeding with `<think>`) or let the model decide.

In [ ]:
class ThinkingModeProcessor:
    """Manages chain-of-thought reasoning for VLM generation.

    Handles three scenarios:
    1. Forced thinking: always prepend <think> to trigger reasoning
    2. Auto thinking: let the model decide based on query complexity
    3. Budget-constrained: limit thinking tokens to control latency
    """

    THINK_OPEN = "<think>"
    THINK_CLOSE = "</think>"

    def __init__(
        self,
        max_thinking_tokens: int = 200,
        thinking_budget: Optional[int] = None,
    ):
        self.max_thinking_tokens = max_thinking_tokens
        self.thinking_budget = thinking_budget or max_thinking_tokens

    def classify_query_complexity(self, query: str) -> str:
        """Heuristic classifier for query complexity.

        In production, this would be a trained classifier or the model
        itself deciding whether to think. Here we use keyword heuristics.
        """
        complex_keywords = [
            "calculate", "compute", "how many", "count", "area",
            "solve", "prove", "derive", "compare", "difference",
            "step by step", "explain why", "reason",
        ]
        query_lower = query.lower()
        if any(kw in query_lower for kw in complex_keywords):
            return "complex"
        return "simple"

    def format_thinking_prompt(self, query: str, force_thinking: bool = False) -> str:
        """Wrap the query to trigger thinking mode."""
        complexity = self.classify_query_complexity(query)

        if force_thinking or complexity == "complex":
            return (
                f"{query}\n\n"
                f"Think step by step before answering. "
                f"Place your reasoning inside {self.THINK_OPEN}...{self.THINK_CLOSE} tags."
            )
        return query

    def parse_thinking_output(self, raw_output: str) -> Dict[str, str]:
        """Separate thinking trace from final answer."""
        if self.THINK_OPEN in raw_output and self.THINK_CLOSE in raw_output:
            think_start = raw_output.index(self.THINK_OPEN) + len(self.THINK_OPEN)
            think_end = raw_output.index(self.THINK_CLOSE)
            thinking = raw_output[think_start:think_end].strip()
            answer = raw_output[think_end + len(self.THINK_CLOSE):].strip()
            return {"thinking": thinking, "answer": answer, "used_thinking": True}

        return {"thinking": "", "answer": raw_output.strip(), "used_thinking": False}

    def estimate_token_budget(self, query: str) -> int:
        """Dynamically allocate thinking budget based on query complexity.

        More complex queries get more thinking tokens.
        This prevents wasting compute on simple questions.
        """
        complexity = self.classify_query_complexity(query)
        budget_map = {"simple": 0, "complex": self.thinking_budget}
        return budget_map[complexity]


# Demonstrate thinking mode
thinker = ThinkingModeProcessor(max_thinking_tokens=200)

# Test queries of different complexity
queries = [
    "What color is the car in this image?",
    "Calculate the total area of all triangles in this diagram.",
    "How many people are sitting vs standing in this photo?",
]

for q in queries:
    complexity = thinker.classify_query_complexity(q)
    budget = thinker.estimate_token_budget(q)
    formatted = thinker.format_thinking_prompt(q)
    print(f"Query: {q}")
    print(f"  Complexity: {complexity}, Budget: {budget} tokens")
    print(f"  Formatted: {formatted[:80]}...")
    print()

# Simulate parsing a thinking-mode output
simulated_output = """<think>
I can see a square with side length 4 and a circle inscribed within it.
Square area = 4 x 4 = 16 square units.
Circle radius = 2, so circle area = pi x 2^2 = 4pi = 12.57.
Shaded area = 16 - 12.57 = 3.43 square units.
</think>
The shaded region has an area of approximately 3.43 square units."""

parsed = thinker.parse_thinking_output(simulated_output)
print("Parsed thinking output:")
print(f"  Used thinking: {parsed['used_thinking']}")
print(f"  Reasoning: {parsed['thinking'][:100]}...")
print(f"  Answer: {parsed['answer']}")

---
# 5) Visual Grounding: Bounding Boxes as Structured Output

## 5.1 Concept

Visual grounding means the model doesn't just describe what it sees — it **localizes** objects by outputting bounding box coordinates. This is critical for verifying the model actually "sees" what it claims (reduces hallucination), enabling downstream tasks, and producing structured API responses.

```
Input:  image + "Locate the dog in this image."
Output: {"object": "dog", "bbox": [0.15, 0.30, 0.65, 0.80], "confidence": 0.92}

Where bbox = [x_min, y_min, x_max, y_max] in normalized [0, 1] coordinates.
```

**Two approaches:**
1. **Tokenized coordinates:** Encode bounding box corners as special tokens in the vocabulary (Qwen2-VL, Kosmos-2)
2. **Text-as-coordinates:** Generate coordinate strings like `<box>(0.15, 0.30, 0.65, 0.80)</box>` and parse them

In [ ]:
class VisualGroundingHead(nn.Module):
    """Predicts bounding boxes from the language model's hidden states.

    When the model generates a special <box> token, this head activates
    to predict (x_min, y_min, x_max, y_max) in normalized coordinates.
    """

    def __init__(self, model_dim: int = 512, num_coord_bins: int = 1000):
        super().__init__()
        self.num_coord_bins = num_coord_bins

        # Approach 1: Regression head (direct coordinate prediction)
        # (batch_num, model_dim) → (batch_num, 4)
        self.regression_head = nn.Sequential(
            nn.Linear(model_dim, model_dim // 2),
            nn.ReLU(),
            nn.Linear(model_dim // 2, 4),
            nn.Sigmoid(),  # Constrain to [0, 1]
        )

        # Approach 2: Classification head (bin-based coordinate prediction)
        # Quantize each coordinate into num_coord_bins discrete bins
        # (batch_num, model_dim) → (batch_num, 4 * num_coord_bins)
        self.classification_head = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.ReLU(),
            nn.Linear(model_dim, 4 * num_coord_bins),
        )

    def forward_regression(self, hidden_state: torch.Tensor) -> torch.Tensor:
        """Direct coordinate regression. Simpler but less precise."""
        # (batch_num, model_dim) → (batch_num, 4)
        return self.regression_head(hidden_state)

    def forward_classification(self, hidden_state: torch.Tensor) -> torch.Tensor:
        """Bin-based coordinate prediction (used by Qwen2-VL).

        Each coordinate is predicted as a classification over bins.
        More precise because the model can learn sharp boundaries.
        """
        # (batch_num, model_dim) → (batch_num, 4 * num_coord_bins)
        logits = self.classification_head(hidden_state)

        # (batch_num, 4, num_coord_bins)
        logits = logits.view(-1, 4, self.num_coord_bins)

        # Take argmax per coordinate and normalize to [0, 1]
        bin_indices = logits.argmax(dim=-1)
        return bin_indices.float() / self.num_coord_bins


class GroundingOutputParser:
    """Parse and validate bounding box outputs from the VLM."""

    @staticmethod
    def parse_text_bbox(text: str) -> Optional[Dict]:
        """Parse bounding box from text like '<box>(0.1, 0.2, 0.5, 0.8)</box>'."""
        import re
        pattern = r"<box>\s*\(([\d.]+),\s*([\d.]+),\s*([\d.]+),\s*([\d.]+)\)\s*</box>"
        match = re.search(pattern, text)
        if match:
            coords = [float(match.group(i)) for i in range(1, 5)]
            return {"bbox": coords, "valid": GroundingOutputParser.validate_bbox(coords)}
        return None

    @staticmethod
    def validate_bbox(bbox: List[float]) -> bool:
        """Check that coordinates are well-formed."""
        x_min, y_min, x_max, y_max = bbox
        in_range = all(0.0 <= c <= 1.0 for c in bbox)
        valid_order = x_min < x_max and y_min < y_max
        min_area = (x_max - x_min) * (y_max - y_min) > 0.001
        return in_range and valid_order and min_area

    @staticmethod
    def compute_iou(bbox_a: List[float], bbox_b: List[float]) -> float:
        """Intersection over Union between two bounding boxes."""
        x1 = max(bbox_a[0], bbox_b[0])
        y1 = max(bbox_a[1], bbox_b[1])
        x2 = min(bbox_a[2], bbox_b[2])
        y2 = min(bbox_a[3], bbox_b[3])

        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area_a = (bbox_a[2] - bbox_a[0]) * (bbox_a[3] - bbox_a[1])
        area_b = (bbox_b[2] - bbox_b[0]) * (bbox_b[3] - bbox_b[1])
        union = area_a + area_b - intersection

        return intersection / union if union > 0 else 0.0


# Demonstrate grounding head
grounding = VisualGroundingHead(model_dim=512).to(DEVICE)
hidden = torch.randn(1, 512, device=DEVICE)

bbox_reg = grounding.forward_regression(hidden)
bbox_cls = grounding.forward_classification(hidden)
print(f"Regression bbox: {bbox_reg.detach().cpu().numpy().round(3)}")
print(f"Classification bbox: {bbox_cls.detach().cpu().numpy().round(3)}")

# Parse text-based output
sample_text = 'The dog is located at <box>(0.15, 0.30, 0.65, 0.80)</box> in the image.'
parsed = GroundingOutputParser.parse_text_bbox(sample_text)
print(f"\nParsed from text: {parsed}")

# IoU computation
pred_bbox = [0.15, 0.30, 0.65, 0.80]
gt_bbox = [0.10, 0.25, 0.60, 0.75]
iou = GroundingOutputParser.compute_iou(pred_bbox, gt_bbox)
print(f"IoU between predicted and ground truth: {iou:.3f}")

## 5.2 Visualizing Grounding Predictions

In [ ]:
def visualize_grounding(
    img_size: Tuple[int, int] = (400, 300),
    predictions: List[Dict] = None,
):
    """Visualize bounding box predictions on a synthetic image."""
    if predictions is None:
        predictions = [
            {"label": "dog", "bbox": [0.1, 0.2, 0.5, 0.7], "conf": 0.92},
            {"label": "ball", "bbox": [0.6, 0.5, 0.8, 0.8], "conf": 0.85},
            {"label": "tree", "bbox": [0.7, 0.05, 0.95, 0.6], "conf": 0.78},
        ]

    # Create a synthetic image with colored regions
    img = np.random.randint(100, 200, (*img_size[::-1], 3), dtype=np.uint8)

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img)

    colors = ["#FF4444", "#44FF44", "#4444FF", "#FFAA00", "#AA00FF"]

    for i, pred in enumerate(predictions):
        bbox = pred["bbox"]
        color = colors[i % len(colors)]

        # Convert normalized coords to pixel coords
        x = bbox[0] * img_size[0]
        y = bbox[1] * img_size[1]
        w = (bbox[2] - bbox[0]) * img_size[0]
        h = (bbox[3] - bbox[1]) * img_size[1]

        rect = patches.Rectangle(
            (x, y), w, h, linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        label_text = f"{pred['label']} ({pred['conf']:.2f})"
        ax.text(
            x, y - 5, label_text, color="white", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8),
        )

    ax.set_title("Visual Grounding: Bounding Box Predictions", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


visualize_grounding()

---
# 6) Evaluation Benchmarks for Vision-Language Models

## 6.1 Overview

Evaluating VLMs requires benchmarks that test multiple capabilities: visual perception, language understanding, reasoning, OCR, and robustness to hallucination.

| Benchmark | Tests | Metric | Top Open Score (2025) |
|-----------|-------|--------|-----------------------|
| MMMU | College-level multimodal reasoning | Accuracy | 72.2 (InternVL3-78B) |
| MMMU-Pro | Harder MMMU with augmented distractors | Accuracy | — |
| MathVista | Math reasoning on figures/charts | Accuracy | Strong (Qwen3-VL) |
| TextVQA | Reading text in natural images (OCR) | Accuracy | — |
| DocVQA | Document understanding | ANLS | 85.6+ |
| POPE | Hallucination detection | F1 | — |
| MMBench | Comprehensive multimodal evaluation | Accuracy | — |
| MM-Vet | Open-ended eval (GPT-4 as judge) | Score | — |

We implement evaluation harnesses for the most important metrics.

In [ ]:
class VLMEvaluator:
    """Evaluation harness for VLM benchmarks.

    Supports multiple-choice accuracy, open-ended generation metrics,
    and hallucination-specific evaluation (POPE protocol).
    """

    # ---- Multiple-Choice Evaluation (MMMU, MMBench) ----

    @staticmethod
    def evaluate_multiple_choice(
        predictions: List[str], ground_truths: List[str]
    ) -> Dict[str, float]:
        """Compute accuracy for multiple-choice VQA.

        Handles common model quirks: extra whitespace, different casing,
        and models that output the full answer text instead of just the letter.
        """
        correct = 0
        total = len(predictions)

        for pred, gt in zip(predictions, ground_truths):
            pred_clean = pred.strip().upper()
            gt_clean = gt.strip().upper()

            # Handle cases where model outputs "A) answer text" or "(A)"
            if len(pred_clean) > 1:
                for char in pred_clean:
                    if char in "ABCDE":
                        pred_clean = char
                        break

            if pred_clean == gt_clean:
                correct += 1

        accuracy = correct / total if total > 0 else 0.0
        return {"accuracy": accuracy, "correct": correct, "total": total}

    # ---- ANLS: Average Normalized Levenshtein Similarity (DocVQA) ----

    @staticmethod
    def _normalized_levenshtein(s1: str, s2: str) -> float:
        """Compute normalized Levenshtein similarity between two strings."""
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        len1, len2 = len(s1), len(s2)

        # Dynamic programming for edit distance
        dp = [[0] * (len2 + 1) for _ in range(len1 + 1)]
        for i in range(len1 + 1):
            dp[i][0] = i
        for j in range(len2 + 1):
            dp[0][j] = j

        for i in range(1, len1 + 1):
            for j in range(1, len2 + 1):
                cost = 0 if s1[i - 1] == s2[j - 1] else 1
                dp[i][j] = min(
                    dp[i - 1][j] + 1,
                    dp[i][j - 1] + 1,
                    dp[i - 1][j - 1] + cost,
                )

        edit_dist = dp[len1][len2]
        return 1.0 - edit_dist / max(len1, len2)

    @staticmethod
    def evaluate_anls(
        predictions: List[str],
        ground_truths: List[List[str]],
        threshold: float = 0.5,
    ) -> Dict[str, float]:
        """Average Normalized Levenshtein Similarity for DocVQA.

        Each question may have multiple acceptable answers. The prediction
        is compared against all of them and the best match is used.
        """
        scores = []
        for pred, gts in zip(predictions, ground_truths):
            pred_lower = pred.strip().lower()
            best_score = max(
                VLMEvaluator._normalized_levenshtein(pred_lower, gt.strip().lower())
                for gt in gts
            )
            scores.append(best_score if best_score >= threshold else 0.0)

        return {"anls": np.mean(scores), "per_sample": scores}

    # ---- POPE: Polling-based Object Probing Evaluation ----

    @staticmethod
    def evaluate_pope(
        predictions: List[str], ground_truths: List[str]
    ) -> Dict[str, float]:
        """POPE evaluation for hallucination detection.

        Questions are yes/no: 'Is there a <object> in the image?'
        Computes accuracy, precision, recall, F1, and the 'yes' ratio
        (a high yes-ratio indicates the model tends to hallucinate objects).
        """
        tp = fp = tn = fn = 0

        for pred, gt in zip(predictions, ground_truths):
            pred_yes = "yes" in pred.strip().lower()
            gt_yes = "yes" in gt.strip().lower()

            if gt_yes and pred_yes:
                tp += 1
            elif gt_yes and not pred_yes:
                fn += 1
            elif not gt_yes and pred_yes:
                fp += 1
            else:
                tn += 1

        total = tp + fp + tn + fn
        accuracy = (tp + tn) / total if total > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        yes_ratio = (tp + fp) / total if total > 0 else 0

        return {
            "accuracy": accuracy, "precision": precision, "recall": recall,
            "f1": f1, "yes_ratio": yes_ratio,
            "confusion": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
        }


evaluator = VLMEvaluator()

In [ ]:
# ---- Demo: Multiple-choice evaluation ----
mc_preds = ["A", "B", "(C)", "A is correct", "D"]
mc_gts = ["A", "B", "C", "A", "C"]
mc_results = evaluator.evaluate_multiple_choice(mc_preds, mc_gts)
print("Multiple-Choice (MMMU-style):")
print(f"  Accuracy: {mc_results['accuracy']:.1%} ({mc_results['correct']}/{mc_results['total']})")

# ---- Demo: ANLS evaluation ----
anls_preds = ["John Smith", "2024-01-15", "$45.99", "Acme Corp"]
anls_gts = [
    ["John Smith", "J. Smith"],
    ["2024-01-15", "January 15, 2024"],
    ["$45.99"],
    ["Acme Corporation", "ACME Corp"],
]
anls_results = evaluator.evaluate_anls(anls_preds, anls_gts)
print(f"\nANLS (DocVQA-style):")
print(f"  ANLS Score: {anls_results['anls']:.3f}")
print(f"  Per-sample: {[f'{s:.2f}' for s in anls_results['per_sample']]}")

# ---- Demo: POPE evaluation ----
pope_preds = ["Yes", "No", "Yes", "Yes", "No", "Yes", "No", "Yes", "No", "No"]
pope_gts = ["Yes", "No", "No", "Yes", "No", "Yes", "Yes", "Yes", "No", "No"]
pope_results = evaluator.evaluate_pope(pope_preds, pope_gts)
print(f"\nPOPE (Hallucination):")
print(f"  Accuracy: {pope_results['accuracy']:.1%}")
print(f"  F1: {pope_results['f1']:.3f}")
print(f"  Precision: {pope_results['precision']:.3f}, Recall: {pope_results['recall']:.3f}")
print(f"  Yes-ratio: {pope_results['yes_ratio']:.1%} (ideal ~ 50%)")

## 6.2 Visualizing Evaluation Results

In [ ]:
def plot_benchmark_comparison():
    """Visualize benchmark scores across hypothetical models."""
    benchmarks = ["MMMU", "MathVista", "DocVQA", "POPE-F1", "TextVQA", "MMBench"]

    # Representative 2025 values for three model families
    models = {
        "Qwen3-VL-72B": [70.5, 75.2, 88.1, 87.3, 82.4, 83.6],
        "InternVL3-78B": [72.2, 71.8, 85.6, 86.1, 80.2, 85.1],
        "LLaVA-OneVision": [65.8, 67.4, 82.3, 84.5, 78.9, 79.2],
    }

    x = np.arange(len(benchmarks))
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ["#4C72B0", "#DD8452", "#55A868"]

    for i, (model_name, scores) in enumerate(models.items()):
        bars = ax.bar(x + i * width, scores, width, label=model_name, color=colors[i])
        for bar, score in zip(bars, scores):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{score:.1f}", ha="center", va="bottom", fontsize=8,
            )

    ax.set_ylabel("Score")
    ax.set_title("VLM Benchmark Comparison (2025)", fontsize=14)
    ax.set_xticks(x + width)
    ax.set_xticklabels(benchmarks)
    ax.legend()
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_benchmark_comparison()

---
# 7) Hallucination Analysis and Mitigation

## 7.1 Taxonomy of VLM Hallucinations

VLMs hallucinate when they generate content that contradicts the visual evidence. This happens because the **language prior** (statistical patterns from text-only training) can overwhelm the **visual signal**.

| Type | Example | Root Cause |
|------|---------|------------|
| **Object** | "There is a cat" (no cat present) | Language priors: cats frequently co-occur with certain scenes |
| **Attribute** | "The red car" (car is blue) | Weak grounding of color/texture features |
| **Relation** | "Dog on the table" (dog is under) | Poor spatial reasoning in the LM |
| **Count** | "Three people" (there are five) | VLMs struggle with counting |
| **Action** | "Person is running" (person is standing) | Temporal assumptions from text |

```
Ground truth scene: A blue car parked next to a tree, with two birds on a fence.

Object hallucination:    "A blue car ... with a dog nearby."
Attribute hallucination: "A RED car parked next to a tree, with two birds."
Relation hallucination:  "A blue car parked UNDER a tree, with two birds ON the car."
Count hallucination:     "A blue car ... with THREE birds."
```

In [ ]:
class HallucinationDetector:
    """Detect and classify hallucinations in VLM outputs.

    Compares model output against ground-truth scene descriptions
    using structured object/attribute/relation extraction.
    """

    @dataclass
    class SceneEntity:
        """An object in the scene with its attributes and relations."""
        name: str
        attributes: List[str]
        relations: List[Tuple[str, str]]  # (relation_type, target_object)
        count: int = 1

    def detect_object_hallucination(
        self, predicted_objects: List[str], ground_truth_objects: List[str]
    ) -> Dict:
        """Find objects the model claims exist but don't."""
        pred_set = {o.lower() for o in predicted_objects}
        gt_set = {o.lower() for o in ground_truth_objects}

        hallucinated = pred_set - gt_set
        missed = gt_set - pred_set
        correct = pred_set & gt_set

        precision = len(correct) / len(pred_set) if pred_set else 1.0
        recall = len(correct) / len(gt_set) if gt_set else 1.0

        return {
            "hallucinated_objects": list(hallucinated),
            "missed_objects": list(missed),
            "correct_objects": list(correct),
            "precision": precision,
            "recall": recall,
        }

    def detect_attribute_hallucination(
        self,
        predicted_attrs: Dict[str, List[str]],
        ground_truth_attrs: Dict[str, List[str]],
    ) -> Dict:
        """Find attribute mismatches (e.g., wrong color, size)."""
        errors = []
        for obj, pred_attrs in predicted_attrs.items():
            if obj in ground_truth_attrs:
                gt_attrs = set(a.lower() for a in ground_truth_attrs[obj])
                for attr in pred_attrs:
                    if attr.lower() not in gt_attrs:
                        errors.append({
                            "object": obj,
                            "hallucinated_attribute": attr,
                            "true_attributes": list(gt_attrs),
                        })
        return {"attribute_errors": errors, "num_errors": len(errors)}

    def compute_chair_metric(
        self,
        generated_objects_per_caption: List[List[str]],
        gt_objects_per_image: List[List[str]],
    ) -> Dict[str, float]:
        """CHAIR (Caption Hallucination Assessment with Image Relevance).

        CHAIRi: fraction of hallucinated objects across all captions
        CHAIRs: fraction of captions containing at least one hallucination
        """
        total_objects = 0
        hallucinated_objects = 0
        captions_with_hallucination = 0

        for pred_objs, gt_objs in zip(generated_objects_per_caption, gt_objects_per_image):
            gt_set = {o.lower() for o in gt_objs}
            caption_has_hallucination = False

            for obj in pred_objs:
                total_objects += 1
                if obj.lower() not in gt_set:
                    hallucinated_objects += 1
                    caption_has_hallucination = True

            if caption_has_hallucination:
                captions_with_hallucination += 1

        chair_i = hallucinated_objects / total_objects if total_objects > 0 else 0
        chair_s = captions_with_hallucination / len(generated_objects_per_caption) if generated_objects_per_caption else 0

        return {"CHAIRi": chair_i, "CHAIRs": chair_s}


detector = HallucinationDetector()

# Object hallucination demo
pred_objects = ["car", "tree", "dog", "bird", "fence"]
gt_objects = ["car", "tree", "bird", "fence", "person"]
obj_results = detector.detect_object_hallucination(pred_objects, gt_objects)
print("Object Hallucination Detection:")
print(f"  Hallucinated: {obj_results['hallucinated_objects']}")
print(f"  Missed: {obj_results['missed_objects']}")
print(f"  Precision: {obj_results['precision']:.2f}, Recall: {obj_results['recall']:.2f}")

# Attribute hallucination demo
pred_attrs = {"car": ["red", "sedan"], "bird": ["small", "black"]}
gt_attrs = {"car": ["blue", "sedan"], "bird": ["small", "brown"]}
attr_results = detector.detect_attribute_hallucination(pred_attrs, gt_attrs)
print(f"\nAttribute Hallucination Detection:")
for err in attr_results["attribute_errors"]:
    print(f"  {err['object']}: predicted '{err['hallucinated_attribute']}', true = {err['true_attributes']}")

# CHAIR metric demo
gen_objects = [
    ["car", "tree", "dog"],       # 'dog' is hallucinated
    ["person", "bench"],           # all correct
    ["cat", "table", "book", "lamp"],  # 'lamp' hallucinated
]
gt_image_objects = [
    ["car", "tree", "bird"],
    ["person", "bench"],
    ["cat", "table", "book"],
]
chair = detector.compute_chair_metric(gen_objects, gt_image_objects)
print(f"\nCHAIR Metrics:")
print(f"  CHAIRi (object-level): {chair['CHAIRi']:.3f}")
print(f"  CHAIRs (caption-level): {chair['CHAIRs']:.3f}")

## 7.2 Mitigation Strategies

We implement three hallucination mitigation techniques:

1. **Self-consistency check** — Generate multiple times and check agreement
2. **Visual grounding verification** — Cross-reference claims against grounding output
3. **Contrastive decoding** — Amplify the difference between the VLM and a text-only LM

In [ ]:
class HallucinationMitigator:
    """Strategies for reducing hallucination in VLM outputs."""

    @staticmethod
    def self_consistency_check(
        responses: List[str], min_agreement: float = 0.6
    ) -> Dict:
        """Generate N responses and return claims that appear in >= min_agreement fraction.

        Intuition: hallucinated details are less stable across samples
        because they come from random language-prior activations,
        while real observations are consistent.
        """
        from collections import Counter

        word_counts = Counter()
        num_responses = len(responses)

        for resp in responses:
            unique_words = set(resp.lower().split())
            for word in unique_words:
                word_counts[word] += 1

        # Filter to content words appearing in sufficient fraction
        stop_words = {
            "the", "a", "an", "is", "are", "in", "on", "at", "of",
            "and", "with", "to", "this", "that",
        }
        consistent_claims = {
            word: count / num_responses
            for word, count in word_counts.items()
            if count / num_responses >= min_agreement
            and word not in stop_words
            and len(word) > 2
        }

        return {"consistent_claims": consistent_claims, "num_responses": num_responses}

    @staticmethod
    def contrastive_decoding(
        vlm_logits: torch.Tensor,
        lm_only_logits: torch.Tensor,
        alpha: float = 0.5,
        beta: float = 0.1,
    ) -> torch.Tensor:
        """Contrastive decoding: amplify what the VLM sees that the LM doesn't.

        final_logits = (1 + alpha) * vlm_logits - alpha * lm_only_logits

        The text-only LM captures pure language bias.
        Subtracting it emphasizes visually-grounded predictions.
        """
        # (batch_num, vocab_size)
        lm_probs = F.softmax(lm_only_logits, dim=-1)

        # Only penalize where LM is confident (above beta threshold)
        lm_confident_mask = (lm_probs > beta).float()

        # Contrastive combination
        adjusted_logits = (1 + alpha) * vlm_logits - alpha * lm_only_logits * lm_confident_mask
        return adjusted_logits

    @staticmethod
    def visual_grounding_verification(
        claimed_objects: List[str],
        grounding_results: List[Dict],
        confidence_threshold: float = 0.3,
    ) -> Dict:
        """Cross-reference text claims against visual grounding.

        For each object the model mentions, check if the grounding head
        can localize it with sufficient confidence.
        """
        grounded_objects = {
            r["label"].lower(): r["confidence"]
            for r in grounding_results
            if r["confidence"] >= confidence_threshold
        }

        verified = []
        suspicious = []

        for obj in claimed_objects:
            if obj.lower() in grounded_objects:
                verified.append({"object": obj, "confidence": grounded_objects[obj.lower()]})
            else:
                suspicious.append({"object": obj, "reason": "not grounded or low confidence"})

        return {
            "verified": verified,
            "suspicious": suspicious,
            "verification_rate": len(verified) / len(claimed_objects) if claimed_objects else 1.0,
        }


mitigator = HallucinationMitigator()

# Self-consistency demo
responses = [
    "A golden retriever is playing with a red ball on the beach.",
    "A golden retriever runs on a sandy beach with a ball.",
    "A golden retriever is on the beach playing fetch with a red ball.",
    "A dog runs on the beach near the water with a ball.",
    "A golden retriever is playing on a sandy beach.",
]
consistency = mitigator.self_consistency_check(responses)
print("Self-Consistency Analysis:")
sorted_claims = sorted(consistency["consistent_claims"].items(), key=lambda x: -x[1])
for word, agreement in sorted_claims[:8]:
    print(f"  '{word}': {agreement:.0%} agreement")

# Contrastive decoding demo
vlm_logits = torch.randn(1, 100)
lm_logits = torch.randn(1, 100)
lm_logits[0, 42] = 5.0   # LM confident about token 42 (potential hallucination)
vlm_logits[0, 42] = 3.0
vlm_logits[0, 77] = 4.0   # VLM confident about token 77 (visual evidence)

adjusted = mitigator.contrastive_decoding(vlm_logits, lm_logits, alpha=0.5)
print(f"\nContrastive Decoding:")
print(f"  Original VLM argmax: {vlm_logits.argmax().item()}")
print(f"  After contrastive:   {adjusted.argmax().item()}")
print(f"  (Token 42 suppressed because LM was overly confident about it)")

# Visual grounding verification demo
claimed = ["dog", "ball", "cat", "beach"]
grounding_res = [
    {"label": "dog", "confidence": 0.92, "bbox": [0.1, 0.2, 0.5, 0.8]},
    {"label": "ball", "confidence": 0.85, "bbox": [0.6, 0.5, 0.7, 0.6]},
    {"label": "cat", "confidence": 0.12, "bbox": [0.0, 0.0, 0.1, 0.1]},
]
verification = mitigator.visual_grounding_verification(claimed, grounding_res)
print(f"\nGrounding Verification:")
print(f"  Verified: {[v['object'] for v in verification['verified']]}")
print(f"  Suspicious: {[s['object'] for s in verification['suspicious']]}")
print(f"  Verification rate: {verification['verification_rate']:.0%}")

## 7.3 Visualizing Hallucination Patterns

In [ ]:
def plot_hallucination_analysis():
    """Visualize the relationship between model confidence and hallucination rate."""
    np.random.seed(42)

    num_samples = 200
    confidence = np.random.beta(2, 1, num_samples)
    hallucination_prob = 0.8 * (1 - confidence) + 0.05
    is_hallucinated = np.random.binomial(1, hallucination_prob)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Plot 1: Confidence distribution by hallucination status
    axes[0].hist(confidence[is_hallucinated == 0], bins=20, alpha=0.7, label="Correct", color="steelblue")
    axes[0].hist(confidence[is_hallucinated == 1], bins=20, alpha=0.7, label="Hallucinated", color="coral")
    axes[0].set_xlabel("Model Confidence")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Confidence vs Hallucination")
    axes[0].legend()

    # Plot 2: Hallucination rate by object type
    object_types = ["Object\nExistence", "Attribute\n(Color/Size)", "Spatial\nRelation", "Count", "Action"]
    hall_rates = [0.15, 0.25, 0.35, 0.45, 0.20]
    colors_bar = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
    axes[1].bar(object_types, hall_rates, color=colors_bar)
    axes[1].set_ylabel("Hallucination Rate")
    axes[1].set_title("Hallucination by Type")
    axes[1].set_ylim(0, 0.6)

    # Plot 3: Effect of mitigation strategies
    strategies = ["Baseline", "Self-\nConsistency", "Contrastive\nDecoding", "Grounding\nVerif.", "All\nCombined"]
    chair_i = [0.35, 0.22, 0.18, 0.15, 0.08]
    axes[2].bar(strategies, chair_i, color="teal", alpha=0.8)
    axes[2].set_ylabel("CHAIRi (lower is better)")
    axes[2].set_title("Mitigation Effectiveness")
    axes[2].set_ylim(0, 0.5)

    for bar_val, x_pos in zip(chair_i, range(len(strategies))):
        axes[2].text(x_pos, bar_val + 0.01, f"{bar_val:.2f}", ha="center", fontsize=9)

    plt.suptitle("Hallucination Analysis Dashboard", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_hallucination_analysis()

---
# 8) Interactive Multimodal Chat with Gradio

## 8.1 Building the Interface

We build a Gradio-based UI that combines all the components we've developed: image upload + text prompt, decoding strategy selection, thinking mode toggle, visual grounding overlay, and streaming output.

**Note:** The model uses random weights, so outputs won't be meaningful. The goal is to demonstrate the *interface and pipeline*, not produce real VLM answers. In practice, you would load pretrained weights (e.g., from Qwen2-VL or LLaVA).

In [ ]:
import gradio as gr
from torchvision import transforms


class MultimodalChatApp:
    """Gradio-based interactive multimodal chat application."""

    def __init__(self, vlm_pipeline: VLMPipeline):
        self.vlm = vlm_pipeline
        self.thinker = ThinkingModeProcessor(max_thinking_tokens=200)
        self.grounding_parser = GroundingOutputParser()

        # Image preprocessing (standard ViT normalization)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        # Simple char-to-id mapping for demo
        self.token_map = {chr(i): i for i in range(256)}

    def _tokenize(self, text: str, max_len: int = 50) -> torch.Tensor:
        """Simple character-level tokenization for demonstration."""
        ids = [self.token_map.get(c, 0) for c in text[:max_len]]
        return torch.tensor([ids], device=DEVICE)

    def _ids_to_text(self, ids: torch.Tensor) -> str:
        """Convert token ids back to text."""
        reverse_map = {v: k for k, v in self.token_map.items()}
        if ids.dim() == 0:
            return ""
        return "".join(reverse_map.get(i.item(), "?") for i in ids)

    def process_query(
        self,
        image: Image.Image,
        prompt: str,
        strategy: str,
        temperature: float,
        thinking_mode: bool,
        max_tokens: int,
    ) -> Tuple[str, str, str]:
        """Process a user query and return (response, thinking_trace, stats)."""
        if image is None:
            return "Please upload an image.", "", ""

        # Preprocess image
        pixel_values = self.transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)

        # Apply thinking mode formatting
        if thinking_mode:
            prompt = self.thinker.format_thinking_prompt(prompt, force_thinking=True)

        # Tokenize and generate
        prompt_ids = self._tokenize(prompt)

        start_time = time.time()
        output_ids = self.vlm.generate(
            pixel_values, prompt_ids,
            max_new_tokens=int(max_tokens),
            temperature=temperature,
            strategy=strategy.lower(),
        )
        elapsed = time.time() - start_time

        # Decode output
        response_text = self._ids_to_text(output_ids)

        # Parse thinking trace if present
        thinking_trace = ""
        if thinking_mode:
            parsed = self.thinker.parse_thinking_output(response_text)
            thinking_trace = parsed["thinking"] or "(No thinking trace in output)"
            response_text = parsed["answer"] or response_text

        # Statistics
        num_tokens = len(output_ids) if output_ids.dim() > 0 else 0
        stats = (
            f"Tokens: {num_tokens} | "
            f"Time: {elapsed:.2f}s | "
            f"Speed: {num_tokens / max(elapsed, 0.001):.1f} tok/s | "
            f"Strategy: {strategy}"
        )

        return response_text, thinking_trace, stats

    def build_interface(self) -> gr.Blocks:
        """Construct the Gradio interface."""
        with gr.Blocks(title="VLM Inference Explorer", theme=gr.themes.Soft()) as demo:
            gr.Markdown("# VLM Inference Explorer\nUpload an image and ask questions. (Demo model with random weights.)")

            with gr.Row():
                with gr.Column(scale=1):
                    image_input = gr.Image(type="pil", label="Input Image")
                    prompt_input = gr.Textbox(label="Prompt", placeholder="Describe this image...")

                    with gr.Row():
                        strategy_dropdown = gr.Dropdown(
                            ["Greedy", "Sampling", "Nucleus", "Top_k"],
                            value="Greedy", label="Decoding Strategy",
                        )
                        temp_slider = gr.Slider(0.1, 2.0, value=0.8, step=0.1, label="Temperature")

                    with gr.Row():
                        thinking_toggle = gr.Checkbox(label="Thinking Mode", value=False)
                        max_tokens_slider = gr.Slider(10, 200, value=50, step=10, label="Max Tokens")

                    submit_btn = gr.Button("Generate", variant="primary")

                with gr.Column(scale=1):
                    response_output = gr.Textbox(label="Response", lines=6)
                    thinking_output = gr.Textbox(label="Thinking Trace", lines=4)
                    stats_output = gr.Textbox(label="Generation Statistics")

            submit_btn.click(
                fn=self.process_query,
                inputs=[image_input, prompt_input, strategy_dropdown, temp_slider, thinking_toggle, max_tokens_slider],
                outputs=[response_output, thinking_output, stats_output],
            )

        return demo


# Build and display the interface
app = MultimodalChatApp(vlm)
demo = app.build_interface()

# Launch in Colab (uncomment to run):
# demo.launch(share=True)
print("Gradio app built successfully. Uncomment demo.launch(share=True) to start the UI.")

## 8.2 Running the Demo

To launch the Gradio interface in Google Colab, uncomment `demo.launch(share=True)` above. This creates a public URL you can share.

Since our model has random weights, the outputs will be gibberish — but the full inference pipeline is fully functional. To get meaningful outputs, you would:

1. **Replace the vision encoder** with a pretrained CLIP-ViT or SigLIP encoder
2. **Replace the LM** with a pretrained model like Qwen2.5 or LLaMA
3. **Load trained projection weights** from LLaVA, Qwen2-VL, or InternVL checkpoints
4. **Use a proper tokenizer** (e.g., from `transformers.AutoTokenizer`)

---
# Summary & Key Takeaways

This notebook covered the complete VLM inference and evaluation stack:

**Pipeline (Section 1):** Image → ViT encoder → MLP projection → concat with text embeddings → causal LM generation. The projection layer bridges vision and language feature spaces, and spatial pooling can trade resolution for speed.

**Decoding (Section 2):** Greedy is fast and deterministic; beam search explores more hypotheses with length normalization; nucleus sampling provides controlled diversity by restricting to a dynamic vocabulary subset.

**Streaming (Section 3):** Python generators enable token-by-token output. KV-caching reduces per-step cost from O(t²) to O(t), essential for production latency.

**Thinking Mode (Section 4):** Chain-of-thought reasoning via `<think>` tags dramatically improves multi-step visual reasoning. Dynamic budget allocation avoids wasting compute on simple queries.

**Visual Grounding (Section 5):** Bounding box prediction (regression or bin-classification) enables verifiable localization. Text-based coordinate parsing offers a simpler alternative.

**Evaluation (Section 6):** MMMU tests college-level reasoning; DocVQA uses ANLS for fuzzy string matching; POPE specifically targets hallucination with yes/no probing.

**Hallucination (Section 7):** Object, attribute, relation, and count hallucinations arise from language priors overwhelming visual evidence. Mitigation via self-consistency, contrastive decoding (subtract LM-only bias), and grounding verification.

**Interactive UI (Section 8):** Gradio provides a rapid prototyping framework that wires together all pipeline components with configurable parameters.